# Experimental Config Path

In [ ]:
import os
from pathlib import Path
exp_path = Path("./experiments")
model_training_time_path = Path("./attrs/nem_utils/logs")
exp_extra = Path("./experiments/extra")
print("Experiment:")
for file in model_training_time_path.iterdir():
        print(file.stem)

Experiment:
nemt
ranem


# Training Time


In [2]:
import numpy as np
for file in model_training_time_path.iterdir():
    print("Experiment: ", file.stem)
    for subfile in file.iterdir():
        data = np.load(subfile / "training_time.npy")
        print(f"{subfile.stem}: {data}")


Experiment:  nemt
ConvNeXt: 360.57445907592773
ResNet: 335.2059941291809
VisionTransformer: 403.8416931629181
VGG: 437.94744849205017
Experiment:  ranem
ConvNeXt: 873.3742117881775
ResNet: 699.0910577774048
VisionTransformer: 913.6929218769073
VGG: 980.8976216316223


# stability

In [3]:
stab_path =  exp_extra / "stability"
[np.mean([float(val) for name, val in  np.load(file / "ranem_ranked/faithfullness.npy")]) for file in stab_path.iterdir()]

[0.6020868028957227,
 0.5813939493377461,
 0.5945212371549363,
 0.5969088031677784,
 0.5820044916700299]

# General results


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Define experiments and initialize list for results
exps = [
    "complexity.npy",
    "faithfullness.npy", 
    "sparseness.npy", 
    #"time.npy"
]

res = []

# Loop through each experiment
for exp_res in exps:    
    for model in (Path(exp_path) / "imagenet").iterdir():  # Adjust the path as needed
        if model.stem == "data":
            continue
        # Loop through each method within the model
        for method in model.iterdir():
            res_path = method / exp_res
            if not res_path.exists():
                print(f"Skipping {res_path} as it does not exist")
                continue
            
                
            data = np.load(res_path)
            if len(data.shape) != 2:
                print(f"Skipping {res_path} as it does not have the correct shape")
                continue
            data = data[:, 1].astype(float)
            # Append experiment, model, method, and the mean & std of the data
            res.append((model.stem, method.stem, res_path.stem, np.nanmean(data), np.nanstd(data)))

# Create DataFrame
res_df = pd.DataFrame(res, columns=["model", "method", "experiment", "mean", "std"])

# Now calculate the mean and std grouped by experiment and method
res_df_mean = res_df.groupby(["experiment", "method"])[["mean", "std"]].agg('mean').reset_index()

# Optionally, you can add a row for the overall average by adding the following:
res_df_mean["model"] = "average"

res_df = pd.concat([res_df, res_df_mean]).reset_index(drop=True)
res_df["mean_std"] = res_df["mean"].round(3).astype(str) + " ± " + res_df["std"].round(3).astype(str)
res_df = res_df.drop(columns=["mean", "std"])

# Create columns for each experiment
res_df = res_df.pivot(index=["model", "method"], columns="experiment", values="mean_std").reset_index()
res_df = res_df[["model", "method", "faithfullness", "complexity",  "sparseness",]]
res_df[res_df["model"] == "average"].drop(columns="model")
# Display the result


Skipping experiments/imagenet/vit_base_patch16_224/iba/complexity.npy as it does not have the correct shape
Skipping experiments/imagenet/vit_base_patch16_224/gradcam/faithfullness.npy as it does not exist
Skipping experiments/imagenet/vit_base_patch16_224/iba/faithfullness.npy as it does not exist
Skipping experiments/imagenet/vit_base_patch16_224/gradcam/sparseness.npy as it does not exist
Skipping experiments/imagenet/vit_base_patch16_224/iba/sparseness.npy as it does not have the correct shape


In [ ]:
res_df[res_df["model"] == "average"].drop(columns="model")

experiment,method,faithfullness,complexity,sparseness
0,gradcam,0.436 ± 0.242,10.322 ± 0.338,0.461 ± 0.095
1,gradshap,0.299 ± 0.229,9.152 ± 0.321,0.842 ± 0.037
2,gradshap_forgrad,0.316 ± 0.26,10.388 ± 0.204,0.479 ± 0.092
3,iba,0.428 ± 0.257,10.318 ± 0.232,0.524 ± 0.111
4,intgrad,0.346 ± 0.239,9.201 ± 0.296,0.837 ± 0.035
5,intgrad_forgrad,0.335 ± 0.259,10.413 ± 0.188,0.468 ± 0.089
6,nemt,0.407 ± 0.241,9.571 ± 0.394,0.7 ± 0.116
7,ranem,0.291 ± 0.254,7.816 ± 0.714,0.85 ± 0.056
8,ranem_ranked,0.469 ± 0.244,10.628 ± 0.0,0.335 ± 0.0
9,rise,0.401 ± 0.249,10.676 ± 0.075,0.284 ± 0.069


In [12]:
res_df[res_df["model"] == "vgg16"].drop(columns="model")

experiment,method,faithfullness,complexity,sparseness
39,gradcam,0.502 ± 0.258,10.533 ± 0.128,0.407 ± 0.087
40,gradshap,0.29 ± 0.224,9.268 ± 0.269,0.829 ± 0.035
41,gradshap_forgrad,0.395 ± 0.284,10.436 ± 0.197,0.456 ± 0.096
42,iba,0.494 ± 0.272,10.354 ± 0.243,0.503 ± 0.117
43,intgrad,0.299 ± 0.229,9.263 ± 0.281,0.83 ± 0.036
44,intgrad_forgrad,0.397 ± 0.286,10.434 ± 0.198,0.457 ± 0.096
45,nemt,0.358 ± 0.271,9.207 ± 0.368,0.795 ± 0.063
46,ranem,0.209 ± 0.264,6.053 ± 0.86,0.99 ± 0.009
47,ranem_ranked,0.397 ± 0.302,10.628 ± 0.0,0.335 ± 0.0
48,rise,0.536 ± 0.252,10.656 ± 0.081,0.307 ± 0.073


In [13]:
res_df[res_df["model"] == "resnet50"].drop(columns="model")

experiment,method,faithfullness,complexity,sparseness
26,gradcam,0.484 ± 0.261,10.344 ± 0.199,0.516 ± 0.101
27,gradshap,0.349 ± 0.261,9.259 ± 0.278,0.831 ± 0.036
28,gradshap_forgrad,0.391 ± 0.315,10.424 ± 0.199,0.465 ± 0.097
29,iba,0.489 ± 0.285,10.234 ± 0.286,0.557 ± 0.124
30,intgrad,0.376 ± 0.269,9.262 ± 0.281,0.83 ± 0.036
31,intgrad_forgrad,0.406 ± 0.31,10.422 ± 0.204,0.465 ± 0.098
32,nemt,0.505 ± 0.28,9.71 ± 0.413,0.68 ± 0.134
33,ranem,0.203 ± 0.299,6.416 ± 1.064,0.983 ± 0.02
34,ranem_ranked,0.582 ± 0.268,10.628 ± 0.0,0.335 ± 0.0
35,rise,0.569 ± 0.258,10.604 ± 0.119,0.348 ± 0.095


In [ ]:
res_df[res_df["model"] == "convnext_small"].drop(columns="model")

experiment,method,faithfullness,complexity,sparseness
13,gradcam,0.322 ± 0.208,10.446 ± 0.203,0.46 ± 0.096
14,gradshap,0.253 ± 0.196,9.204 ± 0.284,0.837 ± 0.036
15,gradshap_forgrad,0.237 ± 0.213,10.411 ± 0.203,0.467 ± 0.094
16,iba,0.301 ± 0.215,10.365 ± 0.167,0.511 ± 0.09
17,intgrad,0.307 ± 0.204,9.235 ± 0.261,0.833 ± 0.033
18,intgrad_forgrad,0.255 ± 0.211,10.438 ± 0.169,0.455 ± 0.085
19,nemt,0.361 ± 0.203,9.485 ± 0.303,0.739 ± 0.077
20,ranem,0.312 ± 0.235,8.507 ± 0.604,0.91 ± 0.046
21,ranem_ranked,0.456 ± 0.191,10.628 ± 0.0,0.335 ± 0.0
22,rise,0.204 ± 0.252,10.717 ± 0.055,0.244 ± 0.058


In [15]:
res_df[res_df["model"] == "vit_base_patch16_224"].drop(columns="model")

experiment,method,faithfullness,complexity,sparseness
52,gradcam,NaN,9.965 ± 0.822,NaN
53,gradshap,0.304 ± 0.234,8.879 ± 0.455,0.872 ± 0.04
54,gradshap_forgrad,0.24 ± 0.228,10.279 ± 0.215,0.529 ± 0.083
55,intgrad,0.402 ± 0.256,9.043 ± 0.363,0.855 ± 0.036
56,intgrad_forgrad,0.284 ± 0.229,10.359 ± 0.182,0.495 ± 0.077
57,nemt,0.403 ± 0.211,9.885 ± 0.491,0.587 ± 0.189
58,ranem,0.44 ± 0.216,10.288 ± 0.329,0.519 ± 0.15
59,ranem_ranked,0.44 ± 0.216,10.628 ± 0.0,0.335 ± 0.0
60,rise,0.294 ± 0.234,10.726 ± 0.043,0.236 ± 0.05
61,saliency,0.325 ± 0.215,9.265 ± 0.842,0.766 ± 0.123


# Robustness

In [ ]:
for model in (Path(exp_path) / "imagenet").iterdir():  # Adjust the path as needed
        if model.stem == "data":
            continue
        # Loop through each method within the model
        for method in model.iterdir():
            res_path = method / "robustness.npy"
            if not res_path.exists():
                print(f"Skipping {res_path} as it does not exist")
                continue
            
                
            data = np.load(res_path)